<a href="https://colab.research.google.com/github/Rafael-Diaz-O/-etflix/blob/main/1Copia_de_Codigo_modificado_image_denoising.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Single Image Denoising Problem**

Image denoising is an inverse problem in which the observed image $\mathbf{y}$ is contaminated by additive noise:

$$
\mathbf{y}=\mathbf{x}+\mathbf{n}, \qquad \mathbf{n}\sim\mathcal{N}(0,\sigma^2).
$$

The goal is to estimate the clean image $\mathbf{x}$ from a noisy observation $\mathbf{y}$.

### **Objective**
Develop a deep-learning model that estimates $\hat{\mathbf{x}}=f_\theta(\mathbf{y})$ for additive white Gaussian noise (AWGN).

### **Experimental protocol**
- Dataset: BSDS500, matching the other image-restoration projects.
- Training crops: $96\times96$ with random crop and horizontal flip.
- Validation crop: centered $288\times288$.
- Fixed noise level: $\sigma=25/255$.
- Training noise is resampled online; validation/test noise is deterministic per image.
- Backbone: same compact residual CNN family used in the other projects.
- Loss: MSE by default.
- Metrics: PSNR and SSIM.
- Validation uses inference mode and the best validation checkpoint is restored.
- The noisy input itself is reported as a baseline.

### **Activities**
1. Improve the baseline performance by changing optimization, loss, capacity, augmentation, or architecture.
2. Compare the restored result against the noisy-input baseline.
3. Document quantitative and visual results in an academic report.


# **Improvement Difficulty Levels**

To improve the baseline model, use the following difficulty guide.

## **Fixed components (cannot be changed)**
- **Dataset:** keep the BSDS500 dataset exactly as provided.
- **Corruption process:** keep the provided denoising corruption process fixed. Do not remove it, replace it, or redesign it.

## **Allowed modifications**
You may improve the model by modifying the training pipeline and/or the neural network, for example:
- hyperparameter tuning
- preprocessing choices
- optimizer and learning-rate strategy
- loss function
- regularization
- weight initialization
- neural network architecture

## **Level 1 — Easy**
Focus on **hyperparameter tuning**. Examples:
- change batch size
- change the number of epochs
- change the learning rate
- change the number of filters
- change the number of residual blocks

## **Level 2 — Medium**
Modify the **training recipe** while keeping the general baseline architecture. Examples:
- compare optimizers (Adam, AdamW, RMSprop, etc.)
- test different losses (MSE, MAE/L1, weighted combinations)
- add regularization such as weight decay or dropout if justified
- improve preprocessing or data augmentation
- test learning-rate scheduling

## **Level 3 — Hard**
Modify the **neural network architecture**. Examples:
- deepen the network
- widen the network
- redesign the residual blocks
- add skip connections
- build a different CNN restoration architecture for the same task

## **Recommendation**
Start with a Level 1 improvement, verify whether PSNR/SSIM improve, and then move to Level 2 or Level 3 if needed. For every modification, explain **what changed**, **why it may help**, and **how the metrics and restored images changed**.


In [ ]:
#@title Import libraries and set reproducibility
!pip -q install torchmetrics

import torch.optim.lr_scheduler as lr_scheduler

import os
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.optim import SGD, Adam, RMSprop
from torchmetrics.functional.image import peak_signal_noise_ratio
from torchmetrics.functional.image import structural_similarity_index_measure

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 17.0 MB/s eta 0:00:00


In [ ]:
#@title Download BSDS500 Training and Validation Dataset
# Same dataset source used by the original Super-Resolution and Inpainting notebooks.
!wget --load-cookies /tmp/cookies.txt "https://docs.google.com/uc?export=download&confirm=$(wget --quiet --save-cookies /tmp/cookies.txt --keep-session-cookies --no-check-certificate 'https://docs.google.com/uc?export=download&id=1qfNMnpY0TI-uyNeUy10KE-NjvU91fFrC' -O- | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1\n/p')&id=1qfNMnpY0TI-uyNeUy10KE-NjvU91fFrC" -O BSDS500.zip && rm -rf /tmp/cookies.txt
!mkdir -p /content/BSDS500
!unzip -q -o BSDS500.zip -d /content/BSDS500

print('Dataset folders:', os.listdir('/content/BSDS500'))


In [ ]:
#1

#@title Utility functions
class AverageMeter: #Calcula los promedios
    def __init__(self):
        self.reset()

    def reset(self): #Resetea los contadores a cero
        self.avg = 0.0
        self.sum = 0.0
        self.count = 0

    def update(self, val, n=1): #Calcula la suma de imagene spor cada lote en cada epoca que al final se usa apra calular el promedio de las metricas como el psnr
        self.sum += float(val) * n
        self.count += n
        self.avg = self.sum / max(self.count, 1)


def read_rgb_file(filename):  #Carga las imagenes
    return Image.open(filename).convert('RGB')


def compute_metrics(pred, target): #Compara que tan buena es la imagen reconstruida con la version original de la imagen
    # Metrics are evaluated in the physical image range [0, 1].
    pred = pred.clamp(0.0, 1.0)
    target = target.clamp(0.0, 1.0)
    psnr = peak_signal_noise_ratio(pred, target, data_range=1.0) # Calcula el PSNR (Peak Signal-to-Noise Ratio), una métrica que mide la relación señal-ruido en decibelios para evaluar la calidad de reconstrucción
    ssim = structural_similarity_index_measure(pred, target, data_range=1.0) #Calcula el SSIM (Structural Similarity Index Measure), una métrica que evalúa la similitud estructural osea el brillo contraste y estrutura y perceptual entre las dos imágenes.
    return psnr.item(), ssim.item()


def show_tensor(img, ax=None, title=None): #Hace las graficas con los valoes, en base a los datos que le llegan de la red neuronal
    img = img.detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    if ax is None:
        plt.imshow(img)
        plt.axis('off')
        if title:
            plt.title(title)
    else:
        ax.imshow(img)
        ax.axis('off')
        if title:
            ax.set_title(title)


In [ ]:
#2
#@title Gaussian noise corruption (fixed noise level)
class CorruptionProcess:
    def __init__(self, sigma=25/255, train=False, seed=1234): # el sigma define al intensidad del ruido
        self.sigma = float(sigma)
        self.train = train
        self.seed = int(seed)

    def __call__(self, x, idx=None): #Recibe las imagenes sin ruido
        # x: (C, H, W), CPU tensor in [0,1]
        if self.train: # si estamos entrenando el modelo
            noise = torch.randn_like(x) * self.sigma # se genera el ruido de forma aleatoria
        else: #Donde se evalua el modelo
            # Deterministic validation/test noise for reproducible metrics.
            g = torch.Generator().manual_seed(self.seed + int(idx or 0))
            noise = torch.randn(x.shape, generator=g, dtype=x.dtype) * self.sigma #Sea segura que al cantidad de ruido en las imagenes sea lamisma
        return (x + noise).clamp(0.0, 1.0)

NOISE_SIGMA = 25 / 255  # Define la constante global del nivel de ruido fijo que se usará en todo el experimento base.


In [ ]:

#@title Dataset loader
class PairedRestorationDataset(Dataset):
    def __init__(self, data_path, transform, corruption):
        self.data_path = data_path
        self.imgs_names = sorted([
            f for f in os.listdir(data_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])
        self.transform = transform
        self.corruption = corruption

    def __len__(self):
        return len(self.imgs_names)

    def __getitem__(self, idx):
        path = os.path.join(self.data_path, self.imgs_names[idx])
        x = self.transform(read_rgb_file(path))
        y = self.corruption(x, idx)
        return x, y


def load_data(data_path, batch_size=4, train_size=96, val_size=288):
    train_transform = transforms.Compose([
        transforms.RandomCrop(train_size),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
    ])
    eval_transform = transforms.Compose([
        transforms.CenterCrop(val_size),
        transforms.ToTensor(),
    ])

    train_corruption = CorruptionProcess(NOISE_SIGMA, train=True, seed=SEED)
    eval_corruption = CorruptionProcess(NOISE_SIGMA, train=False, seed=SEED + 10000)

    train_dir = os.path.join(data_path, 'train')
    val_dir = os.path.join(data_path, 'val')
    test_dir = os.path.join(data_path, 'test')

    train_ds = PairedRestorationDataset(train_dir, train_transform, train_corruption)
    val_ds = PairedRestorationDataset(val_dir, eval_transform, eval_corruption)
    test_ds = PairedRestorationDataset(test_dir, eval_transform, eval_corruption) if os.path.isdir(test_dir) else None

    loader_kwargs = dict(num_workers=2, pin_memory=torch.cuda.is_available())
    g = torch.Generator().manual_seed(SEED)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator=g, **loader_kwargs)
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, **loader_kwargs) if test_ds is not None else None
    return train_loader, val_loader, test_loader


In [ ]:
#3
#@title Deep Neural Network Architecture
class ResidualBlock(nn.Module): #la clase base para todas las redes neuronales en PyTorch
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, 1, 1, bias=True)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=True)

    def forward(self, x):
        residual = x
        out = self.relu(self.conv1(x))
        out = self.conv2(out)
        return out + residual


class DeepNeuralNetwork(nn.Module):  #La red neuronal principal
    """Same compact restoration backbone used by the inpainting baseline."""
    def __init__(self, num_channels=3, base_filter=16, num_residuals=4): #Constructor del modelo. Por defecto recibe 3 canales de color (RGB), 16 filtros base  y 4 bloques residuales
        super().__init__()
        self.conv1 = nn.Conv2d(
            num_channels, base_filter, kernel_size=3, stride=1, #Capa de entrada: Toma la imagen con ruido de 3 canales (RGB) y la expande a un espacio interno de características con un tamaño de base_filter.
            padding=1, bias=True, padding_mode='reflect'
        )
        self.relu = nn.ReLU(inplace=True)
        self.residuals = nn.Sequential(*[
            ResidualBlock(base_filter, base_filter)
            for _ in range(num_residuals)
        ])
        self.conv2 = nn.Conv2d(base_filter, num_channels, kernel_size=1, stride=1, padding=0, bias=True) #Una convolución de tamaño 1x1 que comprime las características internas de vuelta a los num_channels originales 3 canales correspondientes a la imagen RGB ya limpia.

    def forward(self, x): #Define el flujo de datos global de toda la red para una imagen de entrada x.
        out = self.relu(self.conv1(x)) #La imagen con ruido entra a la red, pasa por la primera convolución y por la activación ReLU.
        out = self.residuals(out) #Ese resultado pasa a través de todos los bloques residuales encadenados para filtrar el ruido y extraer patrones limpios.
        return self.conv2(out) #Pasa por la capa convolucional final de 1x1 para producir la imagen restaurada final y la devuelve como salida.


        #--------------------------------------------------------------------------------------------------------------------------------------------------------------------------

class ResNetBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1, padding_mode='reflect')
        self.bn1 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1, padding_mode='reflect')
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return out + residual

class ResNet18StyleDenoising(nn.Module):
    """
    Arquitectura inspirada en ResNet18 pero adaptada para restauración de imágenes
    (sin pérdida de resolución espacial por pooling).
    """
    def __init__(self, num_channels=3, base_filters=64, num_blocks=8):
        super().__init__()
        self.head = nn.Sequential(
            nn.Conv2d(num_channels, base_filters, kernel_size=3, stride=1, padding=1, padding_mode='reflect'),
            nn.ReLU(inplace=True)
        )

        # Bloques residuales tipo ResNet
        blocks = [ResNetBlock(base_filters) for _ in range(num_blocks)]
        self.body = nn.Sequential(*blocks)

        self.tail = nn.Sequential(
            nn.Conv2d(base_filters, base_filters, kernel_size=3, stride=1, padding=1, padding_mode='reflect'),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_filters, num_channels, kernel_size=3, stride=1, padding=1, padding_mode='reflect')
        )

    def forward(self, x):
        h = self.head(x)
        body_out = self.body(h)
        out = self.tail(body_out)
        # Conexión residual global (Image - Noise learning)
        return x + out

      #-----------------------------------------------------------------------------------------------------------
      #U-net
import torch.nn as nn
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, padding_mode='reflect'),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, padding_mode='reflect'),
            nn.BatchNorm2d(channels)
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(x + self.block(x))


class ResidualUNetDenoising(nn.Module):
    def __init__(self, num_channels=3, base_filters=64):
        super().__init__()

        # Encoder
        self.inc = nn.Sequential(
            nn.Conv2d(num_channels, base_filters, kernel_size=3, padding=1, padding_mode='reflect'),
            nn.ReLU(inplace=True)
        )
        self.res1 = ResidualBlock(base_filters)
        self.down1 = nn.Conv2d(base_filters, base_filters * 2, kernel_size=4, stride=2, padding=1)

        self.res2 = ResidualBlock(base_filters * 2)
        self.down2 = nn.Conv2d(base_filters * 2, base_filters * 4, kernel_size=4, stride=2, padding=1)

        # Bottleneck
        self.bottleneck = nn.Sequential(
            ResidualBlock(base_filters * 4),
            ResidualBlock(base_filters * 4)
        )

        # Decoder 1 (Entrada concatenada: 128 + 128 = 256 canales)
        self.up1 = nn.ConvTranspose2d(base_filters * 4, base_filters * 2, kernel_size=2, stride=2)
        self.decode1_res = ResidualBlock(base_filters * 4)
        self.conv_decode1 = nn.Conv2d(base_filters * 4, base_filters * 2, kernel_size=3, padding=1, padding_mode='reflect')

        # Decoder 2 (Entrada concatenada: 64 + 64 = 128 canales)
        self.up2 = nn.ConvTranspose2d(base_filters * 2, base_filters, kernel_size=2, stride=2)
        self.decode2_res = ResidualBlock(base_filters * 2)
        self.conv_decode2 = nn.Conv2d(base_filters * 2, base_filters, kernel_size=3, padding=1, padding_mode='reflect')

        # Salida
        self.outc = nn.Sequential(
            nn.Conv2d(base_filters, base_filters, kernel_size=3, padding=1, padding_mode='reflect'),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_filters, num_channels, kernel_size=3, padding=1, padding_mode='reflect')
        )

    def forward(self, x):
        # Encoder
        x1 = self.inc(x)             # [B, 64, H, W]
        x1_res = self.res1(x1)       # [B, 64, H, W]

        d1 = self.down1(x1_res)      # [B, 128, H/2, W/2]
        d1_res = self.res2(d1)       # [B, 128, H/2, W/2]

        d2 = self.down2(d1_res)      # [B, 256, H/4, W/4]

        # Bottleneck
        bn = self.bottleneck(d2)     # [B, 256, H/4, W/4]

        # Decoder 1
        u1 = self.up1(bn)            # [B, 128, H/2, W/2]
        cat1 = torch.cat([u1, d1_res], dim=1) # [B, 256, H/2, W/2]
        dec1 = self.decode1_res(cat1)
        dec1 = self.conv_decode1(dec1) # [B, 128, H/2, W/2]

        # Decoder 2
        u2 = self.up2(dec1)          # [B, 64, H, W]
        cat2 = torch.cat([u2, x1_res], dim=1) # [B, 128, H, W]
        dec2 = self.decode2_res(cat2) # [B, 128, H, W]
        dec2 = self.conv_decode2(dec2) # [B, 64, H, W]

        # Salida final (Residual learning: resta el ruido estimado)
        out = self.outc(dec2)
        return x - out


In [ ]:
#4
#@title Model hyperparameters, optimizer, and loss
EPOCHS = 150 #@param
BATCH_SIZE = 4 #@param

base_filter = 64 #@param
num_residuals = 12 #@param

model = ResidualUNetDenoising(num_channels=3, base_filters=64).to(device)
#model = ResNet18StyleDenoising(num_channels=3, base_filters=base_filter, num_blocks=num_residuals).to(device)
#model = DeepNeuralNetwork(base_filter=base_filter, num_residuals=num_residuals).to(device)

optimizer_name = 'Adam' #@param ["SGD", "Adam", "RMSprop"] {type:"string"}
cost_function = 'MAE' #@param ["MSE", "MAE"] {type:"string"}
learning_rate = 5e-4 #@param


if optimizer_name == 'SGD':
    optimizer = SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=1e-9)
elif optimizer_name == 'Adam':
    optimizer = Adam(model.parameters(), lr=learning_rate, weight_decay=1e-12)
elif optimizer_name == 'RMSprop':
    optimizer = RMSprop(model.parameters(), lr=learning_rate, weight_decay=1e-9)
else:
    raise ValueError('Unknown optimizer')

criterion = nn.MSELoss() if cost_function == 'MSE' else nn.L1Loss()
#Prueba
scheduler = lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')

CHECKPOINT_PATH = "/content/best_denoising_model.pt"


In [ ]:
#@title Load dataset and visualize samples
data_path = '/content/BSDS500'
train_loader, val_loader, test_loader = load_data(data_path, batch_size=BATCH_SIZE)

x, y = next(iter(train_loader))
num_imgs = min(4, x.size(0))
fig, axs = plt.subplots(2, num_imgs, figsize=(14, 7))
for i in range(num_imgs):
    show_tensor(x[i], axs[0, i], 'Clean target' if i == 0 else None)
    show_tensor(y[i], axs[1, i], 'Noisy' if i == 0 else None)
plt.tight_layout()
plt.show()

print(f'Noise sigma: {NOISE_SIGMA:.6f} = {NOISE_SIGMA*255:.1f}/255')
print('Train batches:', len(train_loader), '| Validation images:', len(val_loader))
print('Independent test set:', 'available' if test_loader is not None else 'not available in this dataset package')


In [ ]:
#5
#@title Baseline vs restored-image evaluation
def evaluate(loader, model=None):
    loss_m, psnr_m, ssim_m = AverageMeter(), AverageMeter(), AverageMeter()

    if model is not None:
        model.eval()

    with torch.inference_mode():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            pred = y if model is None else model(y)
            loss = criterion(pred, x)
            psnr_val, ssim_val = compute_metrics(pred, x)

            n = x.size(0)
            loss_m.update(loss.item(), n)
            psnr_m.update(psnr_val, n)
            ssim_m.update(ssim_val, n)

    return {'loss': loss_m.avg, 'psnr': psnr_m.avg, 'ssim': ssim_m.avg}

# Baseline before training (validation)
baseline_pre = evaluate(val_loader, model=None)
print(f"Noisy-input validation baseline | PSNR {baseline_pre['psnr']:.2f} dB | SSIM {baseline_pre['ssim']:.3f}")


In [ ]:
#6
#@title Training process with validation and best-model checkpoint
history = {
    'train_loss': [], 'train_psnr': [], 'train_ssim': [],
    'val_loss': [], 'val_psnr': [], 'val_ssim': []
}

best_val_psnr = -float('inf')
best_state = None

for epoch in range(EPOCHS):
    losses, psnr_m, ssim_m = AverageMeter(), AverageMeter(), AverageMeter()
    val_losses, val_psnr_m, val_ssim_m = AverageMeter(), AverageMeter(), AverageMeter()

    model.train()
    for x, y in train_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        x_hat = model(y)
        loss = criterion(x_hat, x)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            psnr_val, ssim_val = compute_metrics(x_hat, x)

        n = x.size(0)
        losses.update(loss.item(), n)
        psnr_m.update(psnr_val, n)
        ssim_m.update(ssim_val, n)

    model.eval()
    with torch.inference_mode():
        for x, y in val_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            x_hat = model(y)

            loss = criterion(x_hat, x)
            psnr_val, ssim_val = compute_metrics(x_hat, x)

            n = x.size(0)
            val_losses.update(loss.item(), n)
            val_psnr_m.update(psnr_val, n)
            val_ssim_m.update(ssim_val, n)

    history['train_loss'].append(losses.avg)
    history['train_psnr'].append(psnr_m.avg)
    history['train_ssim'].append(ssim_m.avg)
    history['val_loss'].append(val_losses.avg)
    history['val_psnr'].append(val_psnr_m.avg)
    history['val_ssim'].append(val_ssim_m.avg)

    if val_psnr_m.avg > best_val_psnr:
        best_val_psnr = val_psnr_m.avg
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, CHECKPOINT_PATH)
  #modificado
    scheduler.step()
    print(
        f'Epoch {epoch+1:03d}/{EPOCHS} | '
        f'Loss {losses.avg:.7f} | PSNR {psnr_m.avg:.2f} | SSIM {ssim_m.avg:.3f} | '
        f'Val Loss {val_losses.avg:.7f} | Val PSNR {val_psnr_m.avg:.2f} | Val SSIM {val_ssim_m.avg:.3f}'
    )

model.load_state_dict(best_state)
print(f'Best validation PSNR: {best_val_psnr:.2f} dB')
print('Best checkpoint:', CHECKPOINT_PATH)


In [ ]:
#@title Training curves
fig, axs = plt.subplots(1, 3, figsize=(16, 4))

axs[0].plot(history['train_loss'], label='Train')
axs[0].plot(history['val_loss'], label='Validation')
axs[0].set_title('Loss')
axs[0].set_xlabel('Epoch')
axs[0].legend()

axs[1].plot(history['train_psnr'], label='Train')
axs[1].plot(history['val_psnr'], label='Validation')
axs[1].set_title('PSNR (dB)')
axs[1].set_xlabel('Epoch')
axs[1].legend()

axs[2].plot(history['train_ssim'], label='Train')
axs[2].plot(history['val_ssim'], label='Validation')
axs[2].set_title('SSIM')
axs[2].set_xlabel('Epoch')
axs[2].legend()

plt.tight_layout()
plt.show()


In [ ]:
#7
#@title Baseline vs restored-image evaluation
def evaluate(loader, model=None):
    loss_m, psnr_m, ssim_m = AverageMeter(), AverageMeter(), AverageMeter()

    if model is not None:
        model.eval()

    with torch.inference_mode():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            pred = y if model is None else model(y)
            loss = criterion(pred, x)
            psnr_val, ssim_val = compute_metrics(pred, x)

            n = x.size(0)
            loss_m.update(loss.item(), n)
            psnr_m.update(psnr_val, n)
            ssim_m.update(ssim_val, n)

    return {'loss': loss_m.avg, 'psnr': psnr_m.avg, 'ssim': ssim_m.avg}

final_loader = test_loader if test_loader is not None else val_loader
final_split = 'TEST' if test_loader is not None else 'VALIDATION (no independent test folder found)'

baseline = evaluate(final_loader, model=None)
restored = evaluate(final_loader, model=model)

print(final_split)
print(f"Input baseline  | Loss {baseline['loss']:.7f} | PSNR {baseline['psnr']:.2f} dB | SSIM {baseline['ssim']:.3f}")
print(f"Restored model  | Loss {restored['loss']:.7f} | PSNR {restored['psnr']:.2f} dB | SSIM {restored['ssim']:.3f}")
print(f"PSNR gain       | {restored['psnr'] - baseline['psnr']:+.2f} dB")


In [ ]:
#8
#@title Visual inference
loader = test_loader if test_loader is not None else val_loader
num_images = min(5, len(loader))
fig, axs = plt.subplots(3, num_images, figsize=(4*num_images, 10))
axs = np.array(axs).reshape(3, num_images)

model.eval()
with torch.inference_mode():
    for i, (x, y) in enumerate(loader):
        if i >= num_images:
            break
        pred = model(y.to(device)).cpu().clamp(0, 1)
        show_tensor(y[0], axs[0, i], 'Noisy input' if i == 0 else None)
        show_tensor(pred[0], axs[1, i], 'Denoised' if i == 0 else None)
        show_tensor(x[0], axs[2, i], 'Clean target' if i == 0 else None)

plt.tight_layout()
plt.show()
